In [1]:

import torch
import torch.nn as nn
import torch.optim as optim
import urllib.request
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(url).read().decode("utf-8")[:20000]

chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}
data = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)

seq_len = 40
x_data = []
y_data = []

for i in range(0, len(data) - seq_len - 1, seq_len):
    x_data.append(data[i:i+seq_len])
    y_data.append(data[i+1:i+seq_len+1])

X = torch.stack(x_data)
Y = torch.stack(y_data)

class LSTMModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 32)
        self.lstm = nn.LSTM(32, 64, batch_first=True)
        self.fc = nn.Linear(64, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        return self.fc(x)

model = LSTMModel(len(chars)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.003)
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    model.train()
    total_loss = 0

    for i in range(0, len(X), 64):
        xb = X[i:i+64].to(device)
        yb = Y[i:i+64].to(device)

        optimizer.zero_grad()
        output = model(xb)
        loss = criterion(output.reshape(-1, len(chars)), yb.reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss / ((len(X)-1)//64+1):.4f}")

def generate_text(start_text="ROMEO:", length=300):
    model.eval()
    input_ids = [char_to_idx.get(c, 0) for c in start_text]
    result = start_text

    with torch.no_grad():
        for _ in range(length):
            x = torch.tensor([input_ids[-seq_len:]], dtype=torch.long).to(device)
            output = model(x)
            probabilities = torch.softmax(output[0, -1] / 0.8, dim=0)
            next_id = torch.multinomial(probabilities, 1).item()
            result += idx_to_char[next_id]
            input_ids.append(next_id)

    return result

print("\nGenerated Text:\n")
print(generate_text())


Device: cpu
Epoch 1, Loss: 3.9284
Epoch 2, Loss: 3.4174
Epoch 3, Loss: 3.2200

Generated Text:

ROMEO:tuuo yR
:Fe.fs me ee Iee eonetdSo r teu m as
n ueRR iftnstoe 'tv

eVeasee nt heu,tgeras miner lle eI lu ura ,AhLs sehhu, , mrcfe bn
c  efg 
,ds fthler wiheulm ynhhe w suor e
ter  ssuAoovmt fmiy ntwit cer yA hws t ati y
e tor h?e,ndty cmirt aweiFyrceee aelh uTg oai g so

e ags nnamy  hn sWt tmermew o
